# Project Overview: Batch Data Cleaning and Transformation with Databricks

This project focuses on cleaning and transforming batch sales data in Databricks to prepare it for analytics. 

The objective is to simulate a real-world data engineering pipeline that addresses common data quality issues such as missing values, duplicate records, and data inconsistencies before making the data available for downstream analysis.

Scenario

You are a data engineer working for a retail organization that consolidates sales data from multiple source systems. 

These raw datasets often contain incomplete, duplicated, or inconsistent records. Your responsibility is to design and implement a reliable batch data processing pipeline that cleans and standardizes the data to ensure it is analytics-ready.

Key Tasks

- Load raw sales data containing missing values and duplicate records.
- Clean the data by handling null values, removing duplicates, and standardizing data formats.
- Persist the cleaned and transformed data into Delta tables to support efficient analytics and querying.
- Validate the data pipeline to ensure data quality, consistency, and correctness.

This project mirrors real-world batch data engineering workflows and demonstrates how Databricks, Spark, and Delta Lake can be used to build robust, production-ready data pipelines for analytics.

## Step 1: Create a Directory to Host the Demo Files

In [0]:
output_directory_path = 'dbfs:/FileStore/tables/consolidated_sales_table'

try:
    dbutils.fs.ls(output_directory_path, True)
    dbutils.fs.rm(output_directory_path,recurse=True)
except:
    print(f"{output_directory_path} does not exist")
    dbutils.fs.mkdirs(output_directory_path)

## Step 2: Prepare Output Directory

In [0]:
import pandas as pd

In [0]:
output_file_path = f'{output_directory_path}/consolidated_sales.csv'
# DBTITLE 1,

## Step 2: Prepare Output Directory

In [0]:
data = {
"ProductID": [101, 102, 103, 104, 105, 106, 101], # Duplicate ProductID (101)
"ProductName": ["Widget", "Gadget", "Thingamajig", "Doohickey", None, "Gizmo", "Widget"], # Missing ProductName
"QuantitySold": [15, None, 35, 45, 50, None, 15], # Missing QuantitySold
"SalesAmount": [150.0, 250.0, 350.0, 450.0, 500.0, 600.0, 150.0],
"TransactionDate": ["2023-01-01", "2023-01-02", "2023-01-03", "2023-01-04", "2023-01-05", "2023-01-06", "2023-01-01"], # Duplicate TransactionDate
"timestamp": [1672531200, 1672617600, 1672704000, 1672790400, 1672876800, None, 1672531200] # Missing timestamp
}

In [0]:
df = pd.DataFrame(data)
csv_data = df.to_csv(index=False)
dbutils.fs.put(output_file_path, csv_data, overwrite=True)

display(dbutils.fs.ls(output_file_path))


## Step 4: Load Data into a Spark DataFrame

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType,FloatType,StringType,LongType, DoubleType

schema = StructType([
StructField("ProductID", IntegerType(), True),
StructField("ProductName", StringType(), True),
StructField("QuantitySold", DoubleType(), True),
StructField("SalesAmount", FloatType(), True),
StructField("TransactionDate", StringType(), True),
StructField("timestamp", LongType(), True)
])

spark.sql("DROP TABLE IF EXISTS schema_lakehouse_revolution_ex1.consolidated_sales_record")

consolidated_df = (
    spark
        .read
        .format('csv')
        .option('header','true')
        .schema(schema)
        .load(output_file_path)
).write.format('delta').mode('overwrite').saveAsTable(f"schema_lakehouse_revolution_ex1.consolidated_sales_record")

## Step 5: Validate Data Loading

In [0]:
spark.sql("select * from schema_lakehouse_revolution_ex1.consolidated_sales_record ").show()

## Step 6: Handle Missing Values

In [0]:
missing_values = (
    spark.sql("""
            SELECT SUM(CASE WHEN ProductName IS NULL THEN 1 ELSE 0 END) AS missing_ProductName,
                   SUM(CASE WHEN QuantitySold IS NULL THEN 1 ELSE 0 END) AS missing_QuantitySold,
                   SUM(CASE WHEN timestamp IS NULL THEN 1 ELSE 0 END) AS missing_timestamp                
                FROM schema_lakehouse_revolution_ex1.consolidated_sales_record
        """
    )
)
missing_values.show()

## Step 7: Calculate the Median for the Available QuantitySold Values

In [0]:
spark.sql("""
        SELECT percentile_approx(QuantitySold, 0.5) AS median
        FROM schema_lakehouse_revolution_ex1.consolidated_sales_record
        WHERE QuantitySold IS NOT NULL
""").show()

> ## Step 8: Update Missing Values

In [0]:
default_median = 10

# set default value
spark.sql(f"UPDATE schema_lakehouse_revolution_ex1.consolidated_sales_record SET QuantitySold = {default_median} WHERE QuantitySold IS NULL")

# add median
median_quantity = spark.sql("""
        SELECT percentile_approx(QuantitySold, 0.5) AS median
        FROM schema_lakehouse_revolution_ex1.consolidated_sales_record
        WHERE QuantitySold IS NOT NULL
""").collect()[0]['median']

spark.sql(f"UPDATE schema_lakehouse_revolution_ex1.consolidated_sales_record SET QuantitySold = {median_quantity} WHERE QuantitySold IS NULL")

# fill missing productname with "unknown"
spark.sql(f"UPDATE schema_lakehouse_revolution_ex1.consolidated_sales_record SET ProductName = 'unknown' WHERE ProductName IS NULL")

# fill missing timestamp
spark.sql(f"UPDATE schema_lakehouse_revolution_ex1.consolidated_sales_record SET timestamp =0 WHERE timestamp IS NULL")

spark.sql("select * from schema_lakehouse_revolution_ex1.consolidated_sales_record ").show()


## Step 9: Analyze the Cleaned Data

In [0]:
query = """
    SELECT COUNT(*) AS total_records,
    SUM(CASE WHEN QuantitySold IS NULL THEN 1 ELSE 0 END) AS missing_values,
    AVG(QuantitySold) AS avg_quantity
    FROM schema_lakehouse_revolution_ex1.consolidated_sales_record
"""

spark.sql(query).show()